In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("../Airlinedataset.csv")

features = [
    "days_to_departure",
    "seats_remaining",
    "historical_demand",
    "competitor_price",
    "booking_velocity",
    "is_weekend",
    "flight_capacity"
]

target = "ticket_price"

X = df[features]
y = df[target]

# Same split used throughout the project
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Train the validated model
model = LinearRegression()
model.fit(X_train, y_train)

print("Explainability model trained successfully.")

Explainability model trained successfully.


In [2]:
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

coefficients

,feature,coefficient,absolute_coefficient
5,is_weekend,495.582423,495.582423
0,days_to_departure,-35.347639,35.347639
4,booking_velocity,34.890477,34.890477
2,historical_demand,17.570188,17.570188
1,seats_remaining,-16.617217,16.617217
6,flight_capacity,8.075709,8.075709
3,competitor_price,0.269973,0.269973


In [3]:
# Flight scenario to explain
flight = {
    "days_to_departure": 10,
    "seats_remaining": 35,
    "historical_demand": 85,
    "competitor_price": 6800,
    "booking_velocity": 25,
    "is_weekend": 1,
    "flight_capacity": 180
}

flight_data = pd.DataFrame([flight])[features]

# Model prediction
predicted_price = model.predict(flight_data)[0]

# Feature contributions
contributions = flight_data.iloc[0] * model.coef_

explanation = pd.DataFrame({
    "feature": features,
    "input_value": flight_data.iloc[0].values,
    "coefficient": model.coef_,
    "contribution": contributions.values
})

explanation["absolute_contribution"] = (
    explanation["contribution"].abs()
)

explanation = explanation.sort_values(
    "absolute_contribution",
    ascending=False
)

print(f"Predicted ticket price: ₹{predicted_price:,.2f}")
explanation

Predicted ticket price: ₹12,233.23


,feature,input_value,coefficient,contribution,absolute_contribution
3,competitor_price,6800,0.269973,1835.813649,1835.813649
2,historical_demand,85,17.570188,1493.466022,1493.466022
6,flight_capacity,180,8.075709,1453.627652,1453.627652
4,booking_velocity,25,34.890477,872.261936,872.261936
1,seats_remaining,35,-16.617217,-581.602610,581.602610
5,is_weekend,1,495.582423,495.582423,495.582423
0,days_to_departure,10,-35.347639,-353.476386,353.476386


In [4]:
intercept = model.intercept_

print(f"Model intercept: ₹{intercept:,.2f}")
print(f"Sum of feature contributions: ₹{contributions.sum():,.2f}")

reconstructed_price = intercept + contributions.sum()

print(f"Reconstructed prediction: ₹{reconstructed_price:,.2f}")

Model intercept: ₹7,017.55
Sum of feature contributions: ₹5,215.67
Reconstructed prediction: ₹12,233.23


In [5]:
explanation_display = explanation[
    ["feature", "input_value", "contribution"]
].copy()

explanation_display["contribution"] = (
    explanation_display["contribution"].round(2)
)

explanation_display

,feature,input_value,contribution
3,competitor_price,6800,1835.81
2,historical_demand,85,1493.47
6,flight_capacity,180,1453.63
4,booking_velocity,25,872.26
1,seats_remaining,35,-581.60
5,is_weekend,1,495.58
0,days_to_departure,10,-353.48


In [6]:
print("Price Explanation")
print("=" * 50)
print(f"Base model intercept: ₹{intercept:,.2f}")
print()

for _, row in explanation_display.iterrows():

    sign = "+" if row["contribution"] >= 0 else "-"

    print(
        f"{row['feature']:<20} "
        f"{sign} ₹{abs(row['contribution']):,.2f}"
    )

print()
print(f"Final predicted price: ₹{predicted_price:,.2f}")

Price Explanation
Base model intercept: ₹7,017.55

competitor_price     + ₹1,835.81
historical_demand    + ₹1,493.47
flight_capacity      + ₹1,453.63
booking_velocity     + ₹872.26
seats_remaining      - ₹581.60
is_weekend           + ₹495.58
days_to_departure    - ₹353.48

Final predicted price: ₹12,233.23


## Explainability Findings

The Linear Regression model was decomposed into an intercept and individual feature contributions for a specific flight state.

The contribution analysis shows how each input variable contributes to the final predicted ticket price.

For the selected scenario:

- Days to departure contributes negatively to the predicted price.
- Fewer remaining seats contribute positively to the predicted price.
- Higher historical demand contributes positively.
- Higher booking velocity contributes positively.
- Weekend status contributes positively.
- Competitor price contributes positively.
- Flight capacity contributes positively according to the fitted model.

The individual contributions sum with the model intercept to reproduce the final prediction.

These contributions explain the model's prediction mathematically, but they should not be interpreted as causal effects. For example, a positive coefficient for competitor price does not establish that increasing a competitor's price would causally increase the airline's ticket price by the coefficient amount.

The explanation is therefore a description of the model's learned prediction function rather than evidence of causal pricing behavior.